<a href="https://colab.research.google.com/github/0xfffddd/Coding/blob/main/MGMT687_Qwenv5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Part B. Qwen VL Feature Engineering
# Methdology: mapping feature with 0-100 scores then quantile 1-10
# Checkpoint and cache is used in case of interrupt.
!pip -q install pandas requests tqdm pillow numpy
import os, re, json, time, hashlib, base64
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import getpass
from typing import Dict, Any, Optional
from PIL import Image
from io import BytesIO

# 0. Config
DATASET_CSV = "/content/dataset_history.csv"
IMAGE_ZIP   = "/content/image.zip"
BASE_DIR    = "/content"
OUT_PATH    = "/content/dataset_history_with_llm_features_0_100.csv"
OUT_PATH_10 = "/content/dataset_history_with_llm_features_1_10.csv"
CACHE_DIR   = "/content/llm_cache_qwen_0100"
BASE_URL = "https://dashscope-us.aliyuncs.com/compatible-mode/v1"
CHAT_URL = f"{BASE_URL}/chat/completions"
MODEL = "qwen3-vl-flash-2025-10-15-us"
# Cost controls
MAX_TOKENS   = 140
TEMPERATURE  = 0.0
MAX_SIDE     = 384
JPEG_QUALITY = 80
# Retry controls
MAX_RETRIES = 6
BASE_SLEEP  = 1.0
# Columns
IMAGE_PATH_COL = "image_path"
TITLE_COL = "title"
PRICE_COL = "price"
LABEL_COL = "ordered"

os.makedirs(CACHE_DIR, exist_ok=True)

# 1. Unzip images
if os.path.exists(IMAGE_ZIP):
    print("Unzipping image.zip to /content ...")
    !unzip -q -o "/content/image.zip" -d "/content"
else:
    print("WARNING: /content/image.zip not found. (skip unzip)")

# 2. API key input
API_KEY = getpass.getpass("Enter your DashScope (US) API Key (input hidden): ").strip()
assert API_KEY, "API Key is empty."
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# 3. Load dataset + normalize paths
assert os.path.exists(DATASET_CSV), f"Dataset not found: {DATASET_CSV}"
df = pd.read_csv(DATASET_CSV)
assert IMAGE_PATH_COL in df.columns, f"Expected '{IMAGE_PATH_COL}' not found."
if TITLE_COL not in df.columns:
    print(f"WARNING: '{TITLE_COL}' not found. Prompt will use empty title.")
if PRICE_COL not in df.columns:
    print(f"WARNING: '{PRICE_COL}' not found. (but we don't score price_reasonableness anyway)")
if LABEL_COL not in df.columns:
    print(f"WARNING: '{LABEL_COL}' not found. (label is optional for feature extraction)")
def normalize_to_abs(path_str: str, base_dir: str = BASE_DIR) -> str:
    p = str(path_str).strip().replace("\\", "/")
    if os.path.isabs(p):
        return p
    return os.path.join(base_dir, p)
df["abs_image_path"] = df[IMAGE_PATH_COL].astype(str).apply(normalize_to_abs)
print("Rows:", len(df))
print("Example paths:", df[[IMAGE_PATH_COL, "abs_image_path"]].head(3).to_dict("records"))

# 4. Feature set (0-100)
FEATURE_KEYS_0100 = [
    "image_clarity",
    "visual_score",
    "trend_score",
    "premium_score",
    "unique_score",
    "youth_score",
    "versatility_score",
]

# 5. Prompt
PROMPT_TEMPLATE = """
You are a strict fashion e-commerce analyst evaluating product images.

Your task is to score visual characteristics of this product compared to a large catalog of typical online fashion products.

Important calibration rules:
- Think in percentiles relative to 100 typical products.
- Most products should fall between 40–60.
- Only the best ~10% should receive scores above 80.
- Only the worst ~10% should receive scores below 20.

Avoid repeating the same numbers.
Use a wide range of values whenever you see meaningful differences.

Scoring guide:
90–100: exceptional, clearly among the best products
75–89: strong, above average
55–74: slightly above average
45–54: typical average
30–44: somewhat weak
10–29: poor
0–9: extremely poor

Evaluate the following aspects from the image:
1) image_clarity: Sharpness, lighting, subject visibility, background cleanliness.
2) visual_score: Overall visual attractiveness and presentation quality.
3) trend_score: How modern or fashionable the style appears.
4) premium_score: How expensive or high-quality the product appears visually.
5) unique_score: How distinctive the design is compared to common items.
6) youth_score: How appealing the style is to younger consumers.
7) versatility_score: How easily the item could be matched with different outfits.

Input product:
Title: {title}

Output rules:
- Return ONLY valid JSON.
- Use EXACTLY these keys with INTEGER values between 0 and 100:

{{
  "image_clarity": 0,
  "visual_score": 0,
  "trend_score": 0,
  "premium_score": 0,
  "unique_score": 0,
  "youth_score": 0,
  "versatility_score": 0
}}

Use numbers that differ by at least 3–5 points when possible.
Do not repeat the same number for many fields unless clearly justified.
Do not output explanations.
""".strip()

# 6. Image -> data URL (resize + JPEG compress)
def image_to_data_url_resized(path: str, max_side: int = MAX_SIDE, quality: int = JPEG_QUALITY) -> str:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

# 7. API call + retry
def build_payload(prompt: str, data_url: str) -> Dict[str, Any]:
    return {
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
    }
def call_with_retry(payload: Dict[str, Any]) -> Dict[str, Any]:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.post(CHAT_URL, headers=HEADERS, json=payload, timeout=120)
            try:
                data = r.json()
            except Exception:
                data = {"raw_text": r.text}
            # fail fast on arrearage
            err = (data or {}).get("error", {})
            if err.get("code") == "Arrearage" or err.get("type") == "Arrearage":
                raise RuntimeError(f"ACCOUNT_ARREARAGE: {err.get('message')}")

            if r.status_code == 200:
                return data
            # transient -> retry
            if r.status_code in (429, 500, 502, 503, 504, 408, 409):
                sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
                time.sleep(sleep_s)
                continue
            raise RuntimeError(f"HTTP {r.status_code}: {data}")
        except RuntimeError as e:
            if "ACCOUNT_ARREARAGE" in str(e):
                raise
            last_err = e
            sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
            time.sleep(sleep_s)
        except Exception as e:
            last_err = e
            sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
            time.sleep(sleep_s)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err}")
def extract_text(resp: Dict[str, Any]) -> str:
    return resp.get("choices", [{}])[0].get("message", {}).get("content", "") or ""

# 8. Parse strict JSON
def parse_json_0100(text: str) -> Dict[str, Optional[int]]:
    if not text:
        raise ValueError("Empty response text")
    t = text.strip()
    t = re.sub(r"```json|```", "", t).strip()
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if m:
        t = m.group(0).strip()
    obj = json.loads(t)
    out = {}
    for k in FEATURE_KEYS_0100:
        v = obj.get(k, None)
        try:
            iv = int(v)
            iv = max(0, min(100, iv))
        except Exception:
            iv = None
        out[k] = iv
    return out

# 9. Cache
def make_cache_key(title, abs_image_path, model_name):
    raw = f"{model_name}|{title}|{abs_image_path}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()
def cache_fp(key: str) -> str:
    return os.path.join(CACHE_DIR, f"{key}.json")
def cache_get(key: str):
    fp = cache_fp(key)
    if os.path.exists(fp):
        with open(fp, "r", encoding="utf-8") as f:
            return json.load(f)
    return None
def cache_set(key: str, obj: dict):
    fp = cache_fp(key)
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

# 10. Resume from OUT_PATH if interrupted
done_set = set()
if os.path.exists(OUT_PATH):
    prev = pd.read_csv(OUT_PATH)
    if "abs_image_path" in prev.columns:
        done_set = set(prev["abs_image_path"].astype(str).tolist())
    print(f"Resume: found {len(done_set)} completed rows in {OUT_PATH}")
def append_rows(rows: list, path: str):
    if not rows:
        return
    df_chunk = pd.DataFrame(rows)
    write_header = not os.path.exists(path)
    df_chunk.to_csv(path, mode="a", header=write_header, index=False)

# 11. Main extraction loop
buffer = []
missing = 0
errors  = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting 0-100 features (Qwen VL)"):
    abs_path = str(row["abs_image_path"])
    if abs_path in done_set:
        continue
    title = str(row.get(TITLE_COL, ""))
    out_row = dict(row)
    out_row["_model"] = MODEL
    out_row["_error"] = None
    out_row["_cached"] = 0
    # cache key
    ck = make_cache_key(title, abs_path, MODEL)
    cached = cache_get(ck)
    if cached is not None:
        out_row.update(cached)
        out_row["_cached"] = 1
    else:
        feats = {k: None for k in FEATURE_KEYS_0100}

        if not os.path.exists(abs_path):
            out_row["_error"] = "missing_file"
            missing += 1
        else:
            prompt = PROMPT_TEMPLATE.format(title=title)
            try:
                data_url = image_to_data_url_resized(abs_path)
                payload = build_payload(prompt, data_url)
                resp = call_with_retry(payload)
                feats = parse_json_0100(extract_text(resp))
            except Exception as e:
                out_row["_error"] = str(e)[:2000]
                errors += 1

        out_row.update(feats)
        cache_set(ck, {k: out_row.get(k) for k in FEATURE_KEYS_0100} | {"_error": out_row["_error"]})
    buffer.append(out_row)
    done_set.add(abs_path)
    # flush periodically
    if len(buffer) >= 50:
        append_rows(buffer, OUT_PATH)
        buffer = []
# final flush
append_rows(buffer, OUT_PATH)
print("\nSaved 0-100 dataset to:", OUT_PATH)
print("Missing files:", missing, "| Errors:", errors)

# 12. Map 0-100 to 1-10 and save
out_all = pd.read_csv(OUT_PATH)
def quantile_to_1_10(series: pd.Series) -> pd.Series:
    s = series.copy()
    # If too many NaNs, keep NaN
    if s.dropna().nunique() < 5:
        return pd.Series([np.nan]*len(s), index=s.index)
    # 10 bins to 9 cut points (0.1..0.9)
    qs = [i/10 for i in range(1, 10)]
    cuts = s.dropna().quantile(qs).values
    def map_one(x):
        if pd.isna(x):
            return np.nan
        # count how many cut points x exceeds
        k = 1
        for c in cuts:
            if x > c:
                k += 1
        return int(max(1, min(10, k)))
    return s.apply(map_one)
for k in FEATURE_KEYS_0100:
    out_all[k.replace("_score","") + "_1_10"] = quantile_to_1_10(out_all[k])
out_all.to_csv(OUT_PATH_10, index=False)
print("Saved 1-10 mapped dataset to:", OUT_PATH_10)

# result quick check
print("\nExample distribution (mapped):")
for col in [c for c in out_all.columns if c.endswith("_1_10")]:
    print(col, out_all[col].value_counts(dropna=False).sort_index().to_dict())

Unzipping image.zip to /content ...
Enter your DashScope (US) API Key (input hidden): ··········
Rows: 331
Example paths: [{'image_path': 'image\\image_0.jpg', 'abs_image_path': '/content/image/image_0.jpg'}, {'image_path': 'image\\image_1.jpg', 'abs_image_path': '/content/image/image_1.jpg'}, {'image_path': 'image\\image_2.jpg', 'abs_image_path': '/content/image/image_2.jpg'}]


Extracting 0-100 features (Qwen VL): 100%|██████████| 331/331 [05:20<00:00,  1.03it/s]


Saved 0-100 dataset to: /content/dataset_history_with_llm_features_0_100.csv
Missing files: 1 | Errors: 0
Saved 1-10 mapped dataset to: /content/dataset_history_with_llm_features_1_10.csv

Example distribution (mapped):
image_clarity_1_10 {1.0: 119, 4.0: 57, 6.0: 147, 10.0: 7, nan: 1}
visual_1_10 {1.0: 134, 5.0: 50, 6.0: 38, 7.0: 87, 10.0: 21, nan: 1}
trend_1_10 {1.0: 61, 2.0: 5, 3.0: 38, 4.0: 188, 9.0: 26, 10.0: 12, nan: 1}
premium_1_10 {1.0: 40, 2.0: 26, 3.0: 39, 4.0: 90, 6.0: 3, 7.0: 46, 8.0: 20, 9.0: 50, 10.0: 16, nan: 1}
unique_1_10 {1.0: 33, 2.0: 33, 3.0: 136, 7.0: 50, 8.0: 29, 9.0: 19, 10.0: 30, nan: 1}
youth_1_10 {1.0: 35, 2.0: 47, 3.0: 24, 4.0: 39, 5.0: 26, 6.0: 27, 7.0: 44, 8.0: 51, 9.0: 8, 10.0: 29, nan: 1}
versatility_1_10 {1.0: 40, 2.0: 28, 3.0: 34, 4.0: 35, 5.0: 62, 7.0: 35, 8.0: 75, 10.0: 21, nan: 1}
